# Séance 8 — Réseaux de neurones et interprétabilité

On poursuit le jeu de données « achat » de la séance 7.

**Fil directeur :** données → architecture → paramètres → prédiction → perte → gradient → optimisation → hyperparamètres → évaluation → interprétation.

Objectifs :
- comprendre un neurone et une couche ;
- calculer le nombre de paramètres ;
- comprendre les activations et la non-linéarité ;
- comprendre perte, gradient, rétropropagation et Adam ;
- utiliser `MLPClassifier` ;
- visualiser l'apprentissage ;
- optimiser l'architecture et les hyperparamètres ;
- introduire permutation importance et SHAP.


## 1. Pourquoi un réseau de neurones ?

La régression logistique utilise un score linéaire :

$
z=\beta_0+\beta_1x_1+\cdots+\beta_px_p.
$

Notre jeu de données comporte volontairement des relations non linéaires. On cherche donc un modèle capable de construire des transformations non linéaires successives.

Un réseau de neurones permet cette construction grâce aux **couches** et aux **fonctions d'activation**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance


In [ ]:
# Jeu de données synthétique repris de la séance 7
np.random.seed(42)
n = 2000

age = np.random.uniform(18, 75, n)
revenu = np.clip(np.random.lognormal(np.log(2800), 0.35, n), 1200, 9000)
anciennete = np.clip(np.random.exponential(3, n), 0, 15)
visites = np.random.poisson(5, n)
distance = np.clip(np.random.gamma(2.5, 4, n), 0.2, 40)

ville = np.random.choice(["Tours","Paris","Lyon","Nantes"], n, p=[.30,.30,.20,.20])
statut = np.random.choice(["Nouveau","Régulier","Fidèle"], n, p=[.35,.45,.20])

df = pd.DataFrame({
    "age":age, "revenu":revenu, "anciennete":anciennete,
    "visites":visites, "distance":distance,
    "ville":ville, "statut":statut
})

effet_revenu = np.clip((revenu-2000)/2500, 0, 1.5)
effet_visites = 1.4/(1+np.exp(-(visites-5)))
effet_age = 1.5*np.exp(-((age-40)**2)/(2*12**2))
effet_distance = -0.09*distance
effet_ville = pd.Series(ville).map({"Tours":.5,"Paris":.2,"Lyon":-.1,"Nantes":0}).to_numpy()
effet_statut = pd.Series(statut).map({"Nouveau":-.4,"Régulier":.2,"Fidèle":.8}).to_numpy()

score = -2.4 + 1.2*effet_revenu + effet_visites + effet_age + effet_distance + effet_ville + effet_statut
prob_achat = 1/(1+np.exp(-score))
df["achat"] = np.random.binomial(1, prob_achat)
df.head()


## 2. Le neurone

Un neurone calcule :

$
z=w_1x_1+\cdots+w_px_p+b
$

puis applique une activation :

$
a=f(z).
$

Schématiquement :

```text
x₁ ── w₁ ──┐
x₂ ── w₂ ──┤
x₃ ── w₃ ──┼──► somme + biais ──► activation ──► a
⋮           │
xₚ ── wₚ ──┘
```

Les **poids** et les **biais** sont les paramètres appris.


In [ ]:
def dessiner_neurone():
    fig, ax = plt.subplots(figsize=(9,4))
    ys = [.78,.58,.38,.18]
    for i,y in enumerate(ys,1):
        ax.scatter(.15,y,s=800)
        ax.text(.15,y,f"$x_{i}$",ha="center",va="center",fontsize=13)
        ax.annotate("",xy=(.50,.48),xytext=(.19,y),
                    arrowprops=dict(arrowstyle="->",lw=1.4))
        ax.text((.19+.50)/2,(y+.48)/2+.03,f"$w_{i}$",fontsize=11)
    ax.scatter(.50,.48,s=1400)
    ax.text(.50,.48,"$\\sum+b$",ha="center",va="center",fontsize=13)
    ax.annotate("",xy=(.80,.48),xytext=(.57,.48),
                arrowprops=dict(arrowstyle="->",lw=1.6))
    ax.text(.685,.56,"activation",ha="center")
    ax.scatter(.90,.48,s=900)
    ax.text(.90,.48,"$a$",ha="center",va="center",fontsize=14)
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
    ax.set_title("Un neurone artificiel")
    plt.show()

dessiner_neurone()


## 3. Une couche et le nombre de paramètres

Supposons :

- 5 entrées ;
- 10 neurones cachés ;
- 1 sortie.

Entre l'entrée et la couche cachée :

$
5	\times 10=50
$

poids + 10 biais = 60 paramètres.

Entre la couche cachée et la sortie :

$
10	\times 1=10
$

poids + 1 biais = 11 paramètres.

Donc :

$
\boxed{60+11=71	\text{ paramètres}}.
$

Pour deux couches successives de tailles $n$ et $m$, le nombre de paramètres est :

$
\boxed{nm+m}
$

(car il y a $nm$ poids et $m$ biais).


In [ ]:
def dessiner_architecture(couches, titre="Architecture du réseau"):
    fig, ax = plt.subplots(figsize=(11,5))
    xs = np.linspace(.08,.92,len(couches))
    for i,(x,n) in enumerate(zip(xs,couches)):
        ys = np.linspace(.15,.85,n)
        for y in ys:
            ax.scatter(x,y,s=350,zorder=3)
        if i < len(couches)-1:
            ys2=np.linspace(.15,.85,couches[i+1])
            for y in ys:
                for y2 in ys2:
                    ax.plot([x+.015,xs[i+1]-.015],[y,y2],lw=.35,alpha=.3)
        label="Entrée" if i==0 else ("Sortie" if i==len(couches)-1 else f"Couche {i}")
        ax.text(x,.96,label,ha="center")
        ax.text(x,.05,f"{n} neurone(s)",ha="center")
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
    ax.set_title(titre)
    plt.show()

dessiner_architecture([5,10,1],"5 entrées → 10 neurones → 1 sortie")
dessiner_architecture([5,30,15,1],"5 → 30 → 15 → 1")


## 4. Plusieurs couches : la complexité augmente vite

Pour :

$
5
\rightarrow20
\rightarrow10
\rightarrow1
$

on obtient :

$
5	\times20+20=120,
$

$
20	\times10+10=210,
$

$
10	\times1+1=11.
$

Donc :

$
\boxed{341	\text{ paramètres}}.
$

Le réseau doit donc apprendre 341 nombres.

On peut automatiser ce calcul.


In [ ]:
def compter_parametres(couches):
    details=[]
    total=0
    for a,b in zip(couches[:-1],couches[1:]):
        poids=a*b
        biais=b
        details.append([a,b,poids,biais,poids+biais])
        total += poids+biais
    return total, pd.DataFrame(details,columns=["Entrée","Sortie","Poids","Biais","Total"])

for arch in [[5,5,1],[5,10,1],[5,30,1],[5,100,1],[5,30,15,1],[5,50,30,10,1]]:
    total, details=compter_parametres(arch)
    print(arch, "→", total, "paramètres")


## 5. Pourquoi une activation non linéaire ?

Si toutes les couches étaient linéaires,

\[
h_1=W_1x+b_1,\qquad h_2=W_2h_1+b_2,
\]

alors \(h_2\) resterait une transformation affine de \(x\).

Empiler des couches linéaires ne suffirait donc pas.

On introduit une activation non linéaire, par exemple :

\[
\operatorname{ReLU}(x)=\max(0,x).
\]

Dans une couche cachée :

\[
h=\operatorname{ReLU}(Wx+b).
\]

C'est cette non-linéarité qui donne au réseau sa capacité à représenter des relations complexes.


In [ ]:
x=np.linspace(-5,5,400)
plt.figure(figsize=(7,4))
plt.plot(x,np.maximum(0,x))
plt.axhline(0,lw=.8); plt.axvline(0,lw=.8)
plt.title("Fonction d'activation ReLU")
plt.xlabel("x"); plt.ylabel("ReLU(x)")
plt.grid(alpha=.2); plt.show()


## 6. Construire le `MLPClassifier`

On reprend le `Pipeline` de la séance 7.

`hidden_layer_sizes=(30,)` signifie une couche cachée de 30 neurones.

`hidden_layer_sizes=(30,15)` signifie deux couches cachées de 30 puis 15 neurones.

`hidden_layer_sizes=(50,30,10)` signifie trois couches cachées.

Le prétraitement reste essentiel : les variables numériques sont imputées et standardisées, les variables catégorielles sont encodées.


In [ ]:
variables_numeriques=["age","revenu","anciennete","visites","distance"]
variables_categorielles=["ville","statut"]

X=df.drop(columns=["achat"])
y=df["achat"]

prep_num=Pipeline([
    ("imputation",SimpleImputer(strategy="median")),
    ("normalisation",StandardScaler())
])
prep_cat=Pipeline([
    ("imputation",SimpleImputer(strategy="most_frequent")),
    ("encodage",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])
preparation=ColumnTransformer([
    ("numerique",prep_num,variables_numeriques),
    ("categorielle",prep_cat,variables_categorielles)
])

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=.25,stratify=y,random_state=42
)

modele_neurone=Pipeline([
    ("preparation",preparation),
    ("classification",MLPClassifier(
        hidden_layer_sizes=(30,),
        max_iter=2000,
        random_state=42
    ))
])

modele_neurone.fit(X_train,y_train)


## 7. Combien de paramètres possède réellement le réseau ?

Après le `ColumnTransformer`, le nombre d'entrées du réseau peut être supérieur au nombre de variables initiales, à cause du one-hot encoding.

`coefs_` contient les matrices de poids et `intercepts_` les vecteurs de biais.

On peut donc compter directement tous les paramètres appris.


In [ ]:
mlp=modele_neurone.named_steps["classification"]
noms=modele_neurone.named_steps["preparation"].get_feature_names_out()

print("Nombre d'entrées après préparation :",len(noms))
print("Dimensions des matrices de poids :")
for i,w in enumerate(mlp.coefs_,1):
    print(i,w.shape)

nombre_parametres=sum(w.size for w in mlp.coefs_)+sum(b.size for b in mlp.intercepts_)
print("Nombre total de paramètres :",nombre_parametres)


## 8. Fonction de perte

Pour une classification binaire, le réseau peut utiliser la log-loss :

$
L=-\sum_i
[y_i\log(\hat p_i)+(1-y_i)\log(1-\hat p_i)].
$


La différence avec la régression logistique est surtout l'architecture :

- régression logistique : score linéaire ;
- réseau : transformations successives non linéaires.

Le réseau cherche les poids et biais qui minimisent cette perte.


## 9. Gradient, rétropropagation et optimisation

Pour un paramètre $w$, on cherche :
$
\frac{\partial L}{\partial w}.
$

Pour tous les paramètres :

$
\nabla L.
$

La **rétropropagation** utilise la règle de dérivation en chaîne pour calculer efficacement ces gradients.

L'**optimiseur** utilise ensuite les gradients pour modifier les paramètres.

Dans une descente de gradient :

$
	\theta_{
mnouveau}
=
	\theta_{
mancien}-\eta
\nabla L(	\theta).
$

Il faut donc distinguer :

- **backpropagation** = calcul des gradients ;
- **optimiseur** = mise à jour des paramètres.

`MLPClassifier` utilise par défaut `solver="adam"`. Adam adapte les mises à jour en utilisant l'information contenue dans les gradients successifs.


## 10. Observer l'apprentissage : la courbe de perte

Le réseau conserve sa courbe de perte dans `loss_curve_`.

Une perte qui diminue montre que l'optimisation améliore progressivement le modèle sur les données d'entraînement.

Mais une faible perte d'entraînement ne garantit pas une bonne généralisation.


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(mlp.loss_curve_)
plt.xlabel("Itération")
plt.ylabel("Loss")
plt.title("Évolution de la fonction de perte")
plt.grid(alpha=.2)
plt.show()


## 11. Architecture, complexité et surapprentissage

On compare plusieurs architectures :

\[
(5,), (10,), (20,), (30,), (50,), (100,), (30,15), (50,30,10).
\]

On mesure le train et le test.

Une architecture trop simple peut sous-apprendre.

Une architecture très complexe peut obtenir une excellente performance d'entraînement mais moins bien généraliser.

**Un réseau plus gros n'est donc pas automatiquement meilleur.**


In [ ]:
architectures=[(5,),(10,),(20,),(30,),(50,),(100,),(30,15),(50,30,10)]
resultats=[]

for arch in architectures:
    modele=Pipeline([
        ("preparation",preparation),
        ("classification",MLPClassifier(
            hidden_layer_sizes=arch,max_iter=2000,random_state=42))
    ])
    modele.fit(X_train,y_train)
    ytr=modele.predict(X_train); yte=modele.predict(X_test)
    m=modele.named_steps["classification"]
    npar=sum(w.size for w in m.coefs_)+sum(b.size for b in m.intercepts_)
    resultats.append({
        "Architecture":str(arch),"Paramètres":npar,
        "Accuracy train":accuracy_score(y_train,ytr),
        "Accuracy test":accuracy_score(y_test,yte),
        "F1 train":f1_score(y_train,ytr),
        "F1 test":f1_score(y_test,yte)
    })

resultats_df=pd.DataFrame(resultats)
resultats_df


In [ ]:
resultats_df.set_index("Architecture")[["Accuracy train","Accuracy test","F1 train","F1 test"]].plot(
    kind="bar",figsize=(12,5)
)
plt.ylim(0,1)
plt.ylabel("Score")
plt.title("Complexité du réseau : train vs test")
plt.xticks(rotation=45,ha="right")
plt.tight_layout(); plt.show()


## 12. Paramètres et hyperparamètres

**Paramètres appris :**
- poids ;
- biais.

**Hyperparamètres choisis avant l'apprentissage :**
- nombre de couches ;
- nombre de neurones ;
- activation ;
- `learning_rate_init` ;
- `alpha` ;
- `batch_size` ;
- etc.

La recherche d'hyperparamètres doit utiliser la validation croisée sur le jeu d'entraînement.


## 13. Première recherche : un seul hyperparamètre

On commence par le nombre de neurones.

Avec 6 architectures et 5 plis de validation croisée :

$
6	\times5=30
$

entraînements.

Cette expérience permet de comprendre ce que fait la validation croisée avant d'utiliser `GridSearchCV`.


In [ ]:
architectures_cv=[(5,),(10,),(20,),(30,),(50,),(100,)]
resultats_cv=[]

for arch in architectures_cv:
    modele=Pipeline([
        ("preparation",preparation),
        ("classification",MLPClassifier(
            hidden_layer_sizes=arch,max_iter=2000,random_state=42))
    ])
    scores=cross_val_score(modele,X_train,y_train,cv=5,scoring="f1")
    resultats_cv.append({
        "Architecture":str(arch),
        "F1 moyen":scores.mean(),
        "Écart-type":scores.std()
    })

cv_df=pd.DataFrame(resultats_cv)
cv_df


In [ ]:
plt.figure(figsize=(9,4))
plt.errorbar(cv_df["Architecture"],cv_df["F1 moyen"],yerr=cv_df["Écart-type"],
             fmt="o-",capsize=4)
plt.ylim(0,1)
plt.ylabel("F1 moyen")
plt.xlabel("Architecture")
plt.title("Validation croisée : choix du nombre de neurones")
plt.grid(alpha=.2); plt.show()


## 14. `GridSearchCV`

Supposons 5 architectures et 3 valeurs de `alpha`.
$
5	\times3=15
$

configurations.

Avec 5 plis :

$
15	\times5=75
$

entraînements.

`GridSearchCV` teste systématiquement toutes les combinaisons.


In [ ]:
modele_base=Pipeline([
    ("preparation",preparation),
    ("classification",MLPClassifier(max_iter=2000,random_state=42))
])

parametres={
    "classification__hidden_layer_sizes":[(10,),(20,),(30,),(50,),(30,15)],
    "classification__alpha":[1e-5,1e-4,1e-3]
}

recherche=GridSearchCV(
    modele_base,param_grid=parametres,
    cv=5,scoring="f1",n_jobs=-1,return_train_score=True
)
recherche.fit(X_train,y_train)

print("Meilleurs paramètres :",recherche.best_params_)
print("Meilleur F1 CV :",recherche.best_score_)


In [ ]:
resultats_grid=pd.DataFrame(recherche.cv_results_)
resultats_grid[[
    "param_classification__hidden_layer_sizes",
    "param_classification__alpha",
    "mean_test_score","std_test_score",
    "mean_train_score","rank_test_score"
]].sort_values("rank_test_score").head(10)


## 15. Pourquoi ne pas utiliser le test pour choisir ?

La séparation correcte est :

```text
DONNÉES
   │
   ├── TRAIN ──► validation croisée ──► choix des hyperparamètres
   │
   └── TEST  ─────────────────────────► évaluation finale
```

Le test doit rester indépendant du processus de sélection.


In [ ]:
meilleur_modele=recherche.best_estimator_
y_pred=meilleur_modele.predict(X_test)

print("Accuracy test :",accuracy_score(y_test,y_pred))
print("Precision test:",precision_score(y_test,y_pred))
print("Recall test   :",recall_score(y_test,y_pred))
print("F1 test       :",f1_score(y_test,y_pred))


## 16. `RandomizedSearchCV`

Si l'on teste 6 architectures × 4 valeurs de `alpha` × 4 taux d'apprentissage :

$
6	\times4	\times4=96
$

configurations, soit 480 entraînements avec 5 plis.

`RandomizedSearchCV` tire seulement un nombre fixé de configurations.

Avec `n_iter=20` :

$
20	\times5=100
$

entraînements.


In [ ]:
parametres_random={
    "classification__hidden_layer_sizes":[(10,),(20,),(30,),(50,),(30,15),(50,30,10)],
    "classification__alpha":[1e-5,1e-4,1e-3,1e-2],
    "classification__learning_rate_init":[1e-4,5e-4,1e-3,1e-2]
}

recherche_random=RandomizedSearchCV(
    modele_base,param_distributions=parametres_random,
    n_iter=20,cv=5,scoring="f1",random_state=42,n_jobs=-1
)
recherche_random.fit(X_train,y_train)

print("Meilleurs paramètres :",recherche_random.best_params_)
print("Meilleur F1 CV :",recherche_random.best_score_)


# 17. Permutation importance

Après avoir choisi le modèle, on peut demander :

> Que se passe-t-il si l'on détruit l'information contenue dans une variable ?

Principe :

1. score normal ;
2. mélange aléatoire d'une variable ;
3. nouveau score ;
4. différence.

$
I_j=S_{
m original}-S_{
m permuté,j}.
$

Une grande diminution signifie que le modèle dépend fortement de cette information.

**Attention : importance prédictive ≠ causalité.**


In [ ]:
perm=permutation_importance(
    meilleur_modele,X_test,y_test,
    scoring="f1",n_repeats=10,random_state=42
)

importance=pd.DataFrame({
    "Variable":X_test.columns,
    "Importance moyenne":perm.importances_mean,
    "Écart-type":perm.importances_std
}).sort_values("Importance moyenne",ascending=True)

importance


In [ ]:
plt.figure(figsize=(9,5))
plt.barh(importance["Variable"],importance["Importance moyenne"],
         xerr=importance["Écart-type"])
plt.axvline(0,lw=.8)
plt.xlabel("Diminution moyenne du F1")
plt.ylabel("Variable")
plt.title("Permutation importance")
plt.tight_layout(); plt.show()


# 19. Bilan

Nous avons construit progressivement :


variables → neurones →couches  → prédiction

Puis :

 perte  → gradient  → rétropropagation  → optimiseur  → mise à jour.

Enfin :

hyperparamètres  → validation croisée  → GridSearchCV / RandomizedSearchCV.


Et pour l'interprétation :

- **permutation importance** → importance globale ;
- **SHAP** → contributions aux prédictions.

### À retenir

1. Les poids et biais sont appris ; l'architecture est choisie.
2. Les activations non linéaires permettent des relations complexes.
3. Plus de neurones signifie généralement plus de paramètres.
4. Backpropagation calcule les gradients ; l'optimiseur met à jour les paramètres.
5. Une bonne performance train ne garantit pas la généralisation.
6. Les hyperparamètres sont choisis sans utiliser le test final.
7. Importance prédictive et causalité sont deux notions différentes.

## Transition 

On peut maintenant appliquer toute la démarche à des données réelles :

**exploration → préparation → modèles → validation croisée → hyperparamètres → test final → interprétation.**
